In [1]:
import os
os.environ["FLAGS_use_mkldnn"] = "0"
os.environ["FLAGS_enable_pir_api"] = "0"

import json, cv2, pytesseract

from src.preprocessing.preprocess import Preprocessing
from ocr.tesseract import TesseractModel, TesseractOCR
from ocr.paddle import Paddle, PaddleModel, PaddleParams
from src.preprocessing.preprocess import Preprocessing

In [2]:
FILE_PATHS = ["data/input/preprocess/1.png", "data/input/preprocess/2.png"]
OUT_DIR = "data/output/test"
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
os.makedirs(OUT_DIR, exist_ok=True)
cfg = json.load(open("config.json")) if os.path.exists("config.json") else {}

In [3]:
prep = Preprocessing(config=cfg)
tesseract = TesseractOCR(model=TesseractModel())
paddle = Paddle(model=PaddleModel(params=PaddleParams(
    lang='en', use_textline_orientation=True, device='cpu'
)))

Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\kunnic\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
c:\Users\kunnic\Desktop\code\VNDigitizeComprehensiveSystem_Team03\.venv\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv5_server_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\kunnic\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('en_PP-OCRv5_mobile_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\kunnic\.paddlex\

In [4]:
prep_images = []
for path in FILE_PATHS:
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(f"Could not read {path}")
    prep_img = prep._process(img).image
    cv2.imwrite(os.path.join(OUT_DIR, f"prep_{os.path.basename(path)}"), prep_img)
    prep_images.append(prep_img)

In [5]:
import time

tess_start = time.perf_counter()
tess_results = tesseract.recognize(prep_images)
tess_end = time.perf_counter()

tess_total_time = tess_end - tess_start
print(f"Tesseract Batch Time : {tess_total_time:.4f} seconds")

padd_start = time.perf_counter()
padd_results = paddle.recognize(prep_images)
padd_end = time.perf_counter()

padd_total_time = padd_end - padd_start
print(f"PaddleOCR Batch Time : {padd_total_time:.4f} seconds")
print("-" * 40)

Tesseract Batch Time : 2.1262 seconds
PaddleOCR Batch Time : 23.0163 seconds
----------------------------------------


In [9]:
os.makedirs(f"{OUT_DIR}/json/tess", exist_ok=True)
os.makedirs(f"{OUT_DIR}/json/padd", exist_ok=True)

for i, result in enumerate(tess_results):
    output_file = f"{OUT_DIR}/json/tess/result_{i}.json"
    
    result.to_json(output_path=output_file)

for i, result in enumerate(padd_results):
    output_file = f"{OUT_DIR}/json/padd/result_{i}.json"
    
    result.to_json(output_path=output_file)

NameError: name 'indent' is not defined